# 🏥 Insurance Claim Amount Prediction — End-to-End Project

Predicts the medical insurance `claim` amount for a policyholder based on
age, BMI, blood pressure, smoking/diabetic status, number of children, and region.

This is a **regression** problem (predicting a continuous dollar amount), not
classification.

**Pipeline**
- EDA (target distribution, feature relationships, correlations)
- Missing value treatment
- Categorical encoding (saved for consistent use at inference time)
- Train/test split
- Compare Linear Regression vs Random Forest vs Gradient Boosting
- Hyperparameter tuning on the best model
- Feature importance
- Save all artifacts needed to deploy a Streamlit app

**Dataset:** `insurance.csv` — columns:
`Id, age, gender, bmi, bloodpressure, diabetic, children, smoker, region, claim`

> Run all cells top to bottom in Google Colab.

## 1. Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

RANDOM_STATE = 42

## 2. Load the Dataset

Upload `insurance.csv` when prompted.

In [ ]:
from google.colab import files
import os

DATA_PATH = '/content/insurance.csv'

if not os.path.exists(DATA_PATH):
    print("Dataset not found at", DATA_PATH)
    print("Please upload insurance.csv now...")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
print("Dataset loaded:", df.shape)
df.head()

## 3. Initial Exploration

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# drop the Id column - it's just a row identifier, not predictive
df = df.drop(columns=['Id'])
df.head()

In [ ]:
df.isnull().sum()

## 4. Missing Value Treatment

In [ ]:
# age: numeric, mild missingness -> fill with median
df['age'] = df['age'].fillna(df['age'].median())

# region: categorical, small number missing -> fill with the most common region
df['region'] = df['region'].fillna(df['region'].mode()[0])

print("Remaining missing values:")
df.isnull().sum()

## 5. Target Distribution — Claim Amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['claim'], bins=40, kde=True, ax=axes[0], color='#38BDF8')
axes[0].set_title('Claim Amount Distribution')

sns.boxplot(x=df['claim'], ax=axes[1], color='#38BDF8')
axes[1].set_title('Claim Amount — Outliers')
plt.tight_layout()
plt.show()

print("Claim amount is right-skewed — a handful of high-cost claims pull the tail out.")

## 6. How Features Relate to Claim Amount

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.boxplot(x='smoker', y='claim', data=df, ax=axes[0, 0], palette=['#2DD4A7', '#FB7185'])
axes[0, 0].set_title('Claim Amount by Smoking Status')

sns.boxplot(x='diabetic', y='claim', data=df, ax=axes[0, 1], palette=['#2DD4A7', '#FB7185'])
axes[0, 1].set_title('Claim Amount by Diabetic Status')

sns.scatterplot(x='bmi', y='claim', hue='smoker', data=df, ax=axes[1, 0], palette=['#2DD4A7', '#FB7185'])
axes[1, 0].set_title('BMI vs Claim Amount (colored by smoker)')

sns.scatterplot(x='age', y='claim', hue='smoker', data=df, ax=axes[1, 1], palette=['#2DD4A7', '#FB7185'])
axes[1, 1].set_title('Age vs Claim Amount (colored by smoker)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x='region', y='claim', data=df, palette='crest')
plt.title('Average Claim Amount by Region')
plt.show()

## 7. Encode Categorical Features

In [ ]:
gender_encoder = LabelEncoder()
diabetic_encoder = LabelEncoder()
smoker_encoder = LabelEncoder()
region_encoder = LabelEncoder()

df['gender'] = gender_encoder.fit_transform(df['gender'])
df['diabetic'] = diabetic_encoder.fit_transform(df['diabetic'])
df['smoker'] = smoker_encoder.fit_transform(df['smoker'])
df['region'] = region_encoder.fit_transform(df['region'])

print("gender:", list(gender_encoder.classes_), "->", list(range(len(gender_encoder.classes_))))
print("diabetic:", list(diabetic_encoder.classes_), "->", list(range(len(diabetic_encoder.classes_))))
print("smoker:", list(smoker_encoder.classes_), "->", list(range(len(smoker_encoder.classes_))))
print("region:", list(region_encoder.classes_), "->", list(range(len(region_encoder.classes_))))

df.head()

## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

## 9. Train/Test Split

In [ ]:
X = df.drop(columns=['claim'])
y = df['claim']

FEATURE_COLUMNS = list(X.columns)
print("Feature columns (in order):", FEATURE_COLUMNS)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train shape:", x_train.shape, "| Test shape:", x_test.shape)

## 10. Model Comparison

In [ ]:
def evaluate_regressor(name, model, X_test, y_test):
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"{name:22s} | RMSE: {rmse:9.2f} | MAE: {mae:9.2f} | R2: {r2:.4f}")
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

In [ ]:
results = []

lin_reg = LinearRegression()
lin_reg.fit(x_train, y_train)
results.append(evaluate_regressor("Linear Regression", lin_reg, x_test, y_test))

rf_reg = RandomForestRegressor(random_state=RANDOM_STATE)
rf_reg.fit(x_train, y_train)
results.append(evaluate_regressor("Random Forest", rf_reg, x_test, y_test))

gb_reg = GradientBoostingRegressor(random_state=RANDOM_STATE)
gb_reg.fit(x_train, y_train)
results.append(evaluate_regressor("Gradient Boosting", gb_reg, x_test, y_test))

results_df = pd.DataFrame(results).set_index('Model')
display(results_df.style.format('{:.2f}').background_gradient(cmap='Blues', subset=['RMSE', 'MAE']))

In [ ]:
results_df[['RMSE', 'MAE']].plot(kind='bar', figsize=(8, 5), rot=0, color=['#38BDF8', '#FBBF24'])
plt.title('Model Comparison — Lower is Better')
plt.ylabel('Error')
plt.show()

results_df[['R2']].plot(kind='bar', figsize=(6, 4), rot=0, color='#2DD4A7', legend=False)
plt.title('Model Comparison — R² (Higher is Better)')
plt.ylabel('R2 Score')
plt.ylim(0, 1)
plt.show()

## 11. Hyperparameter Tuning

Tune the best-performing model from above (default: Random Forest — change
`base_estimator` if Gradient Boosting won on your run).

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 8, 15],
    'min_samples_leaf': [1, 2, 5],
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
grid_search.fit(x_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV RMSE:", -grid_search.best_score_)

In [ ]:
rf_tuned = grid_search.best_estimator_
tuned_result = evaluate_regressor("Random Forest (tuned)", rf_tuned, x_test, y_test)

## 12. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': FEATURE_COLUMNS,
    'Importance': rf_tuned.feature_importances_
}).sort_values('Importance', ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance — Tuned Random Forest')
plt.tight_layout()
plt.show()

## 13. Save Model Artifacts

Pick whichever model performed best on your run.

In [ ]:
final_model = rf_tuned  # swap to lin_reg / gb_reg / rf_reg if a different model won

joblib.dump(final_model, 'claim_model.pkl')
joblib.dump(gender_encoder, 'gender_encoder.pkl')
joblib.dump(diabetic_encoder, 'diabetic_encoder.pkl')
joblib.dump(smoker_encoder, 'smoker_encoder.pkl')
joblib.dump(region_encoder, 'region_encoder.pkl')
joblib.dump(FEATURE_COLUMNS, 'feature_columns.pkl')

print("Saved artifacts:")
print(" - claim_model.pkl")
print(" - gender_encoder.pkl")
print(" - diabetic_encoder.pkl")
print(" - smoker_encoder.pkl")
print(" - region_encoder.pkl")
print(" - feature_columns.pkl")

## 14. Download Artifacts (for the Streamlit App)

In [ ]:
from google.colab import files

for f in ['claim_model.pkl', 'gender_encoder.pkl', 'diabetic_encoder.pkl',
          'smoker_encoder.pkl', 'region_encoder.pkl', 'feature_columns.pkl']:
    files.download(f)

## 🚀 Next Steps — Deploying to Streamlit

1. Download the 6 files above.
2. Put them alongside `app.py`, `requirements.txt`, and `.streamlit/config.toml`.
3. Push to a public GitHub repo.
4. Deploy on [share.streamlit.io](https://share.streamlit.io) — main file `app.py`.

See the accompanying `README.md` for the full step-by-step guide.